# Ejercicio 5: Espacio Vectorial

### Nombre: Kevin Alvear

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


In [69]:
import pandas as pd

# Cargar dataset (ajusta la ruta)
df = pd.read_csv("data/wikipedia_text_corpus.csv")

# Ver estructura
df.head()

,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


In [70]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # quitar símbolos
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [stemmer.stem(w) for w in tokens]
    return " ".join(tokens)

# Aplicar al corpus
df['processed'] = df['text'].apply(preprocess)

[nltk_data] Downloading package stopwords to C:\Users\Kevin
[nltk_data]     Alvear\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(df['processed'])

In [ ]:
tfidf_matrix.shape

(10859, 134976)

In [ ]:
queries = [
    "battery indicator in cars",
    "computer services company",
    "voltage measurement in batteries",
    "analog integrated circuits",
    "mobile battery status",
    "electrical charge systems",
    "trustpilot company rating",
    "electronic devices battery",
    "voltmeter and ammeter",
    "integrated circuit design"
]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def search_tfidf(query, top_k=5):
    query_vec = vectorizer.transform([query])
    
    scores = cosine_similarity(query_vec, tfidf_matrix)[0]
    
    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]
    
    results = []
    
    for rank, idx in enumerate(top_indices):
        results.append({
            "rank": rank + 1,
            "doc_id": idx,
            "score": scores[idx],
            "text": df.iloc[idx]['text'][:200]
        })
    
    return results

In [ ]:
import pandas as pd

for q in queries:
    print("\n" + "="*90)
    print("QUERY:", q)
    
    results = search_tfidf(q)
    
    table = pd.DataFrame(results)
    print(table)


QUERY: battery indicator in cars
   rank  doc_id     score                                               text
0     1    7778  0.168730  Pressure reference system\n\nPressure referenc...
1     2   10102  0.153645  Nav/attack system\n\nA nav/attack system (shor...
2     3   10508  0.104865  USHUS (sonar)\n\nUSHUS is an integrated sonar ...
3     4    4113  0.077186  Dimethyl dicarbonate\n\nDimethyl dicarbonate (...
4     5   10260  0.048345  Tablet computer\n\nA tablet computer, commonly...

QUERY: computer services company
   rank  doc_id     score                                               text
0     1    3578  0.263324  Improvement Support Systems\n\nImprovement Sup...
1     2    6376  0.152894  Delivery bar code sorter\n\nDelivery Bar Code ...
2     3    4133  0.125503  Paper-ruling machine\n\nA paper-ruling machine...
3     4    5121  0.112545  Scandinavian Multi Access Reservations for Tra...
4     5    7319  0.099201  House R 128\n\nHouse R 128 (Sobek House) is a ...

QUERY: 

## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [ ]:
from rank_bm25 import BM25Okapi

In [ ]:
# corpus tokenizado
tokenized_corpus = [doc.split() for doc in df['processed']]

In [ ]:
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
def search_bm25(query, top_k=5):
    tokenized_query = query.lower().split()
    
    scores = bm25.get_scores(tokenized_query)
    
    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]
    
    results = []
    
    for rank, idx in enumerate(top_indices):
        results.append({
            "rank": rank + 1,
            "doc_id": idx,
            "score": scores[idx],
            "text": df.iloc[idx]['text'][:200]
        })
    
    return results

In [ ]:
queries = [
    "battery indicator in cars",
    "computer services company",
    "voltage measurement in batteries",
    "analog integrated circuits",
    "mobile battery status",
    "electrical charge systems",
    "trustpilot company rating",
    "electronic devices battery",
    "voltmeter and ammeter",
    "integrated circuit design"
]

In [ ]:
import pandas as pd

for q in queries:
    print("\n" + "="*90)
    print("QUERY:", q)
    
    results = search_bm25(q)
    
    table = pd.DataFrame(results)
    print(table)


QUERY: battery indicator in cars
   rank  doc_id      score                                               text
0     1   10102  11.673276  Nav/attack system\n\nA nav/attack system (shor...
1     2    7778  11.632838  Pressure reference system\n\nPressure referenc...
2     3   10508  11.166024  USHUS (sonar)\n\nUSHUS is an integrated sonar ...
3     4    4113   9.329384  Dimethyl dicarbonate\n\nDimethyl dicarbonate (...
4     5    7150   7.286950  AN/ALR-67 Radar Warning Receiver\n\nThe AN/ALR...

QUERY: computer services company
   rank  doc_id     score                                               text
0     1    3578  8.514954  Improvement Support Systems\n\nImprovement Sup...
1     2    5121  8.466553  Scandinavian Multi Access Reservations for Tra...
2     3    6376  8.252767  Delivery bar code sorter\n\nDelivery Bar Code ...
3     4    7069  7.880585  George Berkeley Ross\n\nGeorge Berkeley Ross (...
4     5    4133  7.837167  Paper-ruling machine\n\nA paper-ruling machine...

Q

## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 

In [ ]:
# Crear un DataFrame para comparar los resultados de ambos modelos
comparison_results = []

for query in queries:
    tfidf_results = search_tfidf(query, top_k=5)
    bm25_results = search_bm25(query, top_k=5)
    
    comparison_results.append({
        'query': query,
        'tfidf_docs': [r['doc_id'] for r in tfidf_results],
        'tfidf_scores': [r['score'] for r in tfidf_results],
        'bm25_docs': [r['doc_id'] for r in bm25_results],
        'bm25_scores': [r['score'] for r in bm25_results]
    })

# Mostrar comparación
for result in comparison_results:
    print("\n" + "="*90)
    print(f"QUERY: {result['query']}")
    print("\nTF-IDF:")
    print(f"  Documentos: {result['tfidf_docs']}")
    print(f"  Scores: {[f'{s:.4f}' for s in result['tfidf_scores']]}")
    print("\nBM25:")
    print(f"  Documentos: {result['bm25_docs']}")
    print(f"  Scores: {[f'{s:.4f}' for s in result['bm25_scores']]}")
    
    # Documentos en común
    common_docs = set(result['tfidf_docs']) & set(result['bm25_docs'])
    print(f"\nDocumentos en común: {common_docs}")


QUERY: battery indicator in cars

TF-IDF:
  Documentos: [7778, 10102, 10508, 4113, 10260]
  Scores: ['0.1687', '0.1536', '0.1049', '0.0772', '0.0483']

BM25:
  Documentos: [10102, 7778, 10508, 4113, 7150]
  Scores: ['11.6733', '11.6328', '11.1660', '9.3294', '7.2869']

Documentos en común: {4113, 7778, 10508, 10102}

QUERY: computer services company

TF-IDF:
  Documentos: [3578, 6376, 4133, 5121, 7319]
  Scores: ['0.2633', '0.1529', '0.1255', '0.1125', '0.0992']

BM25:
  Documentos: [3578, 5121, 6376, 7069, 4133]
  Scores: ['8.5150', '8.4666', '8.2528', '7.8806', '7.8372']

Documentos en común: {6376, 5121, 3578, 4133}

QUERY: voltage measurement in batteries

TF-IDF:
  Documentos: [7778, 10102, 10508, 4113, 10260]
  Scores: ['0.1687', '0.1536', '0.1049', '0.0772', '0.0483']

BM25:
  Documentos: [10102, 7778, 10508, 4113, 7150]
  Scores: ['11.6733', '11.6328', '11.1660', '9.3294', '7.2869']

Documentos en común: {4113, 7778, 10508, 10102}

QUERY: analog integrated circuits

TF-IDF:
  